#### 1. Define the raw path and explicit string schemas

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
)

RAW = "Files/buildmate_raw_data/raw"


def string_schema(*column_names):
    """
    Create a schema in which every source column
    is stored as a nullable string.
    """
    return StructType([
        StructField(
            column_name,
            StringType(),
            True,
        )
        for column_name in column_names
    ])


customers_schema = string_schema(
    "CUSTOMER_ID",
    "CUSTOMER_NAME",
    "REGISTERED_ON",
    "KYC_VERIFIED_ON",
    "CUSTOMER_TYPE",
    "CITY",
)

rentals_schema = string_schema(
    "rental_id",
    "customer_id",
    "depot_code",
    "asset_id",
    "checkout_ts",
    "checkin_ts",
    "rental_type",
)

billing_schema = string_schema(
    "BILL_ID",
    "RENTAL_ID",
    "BILL_DATE",
    "AMOUNT_INR",
    "PAYER_TYPE",
)

depots_schema = string_schema(
    "DEPOT_CODE",
    "DEPOT_NAME",
    "ZONE",
    "FLEET_SIZE",
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 3, Finished, Available, Finished, False)

#### 2. Create the Bronze helper functions

In [2]:
def read_raw_csv(path, schema):
    """
    Read a raw CSV using an explicit all-string schema.
    """
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .schema(schema)
        .load(path)
    )


def to_bronze(df):
    """
    Add lineage only.

    No trimming, parsing, casting, deduplication,
    standardisation, or filtering occurs here.
    """
    return (
        df
        .withColumn(
            "ingested_at",
            F.current_timestamp(),
        )
        .withColumn(
            "source_file",
            F.input_file_name(),
        )
    )

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 4, Finished, Available, Finished, False)

#### 3. Read all four sources

##### Customers

In [3]:
bronze_customers_df = to_bronze(
    read_raw_csv(
        f"{RAW}/customer_master/customer_master_export.csv",
        customers_schema,
    )
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 5, Finished, Available, Finished, False)

##### Depots

In [4]:
bronze_depots_df = to_bronze(
    read_raw_csv(
        f"{RAW}/depots/depot_master.csv",
        depots_schema,
    )
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 6, Finished, Available, Finished, False)

##### Billing

In [5]:
bronze_billing_df = to_bronze(
    read_raw_csv(
        f"{RAW}/billing/billing_export.csv",
        billing_schema,
    )
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 7, Finished, Available, Finished, False)

##### Rentals — one wildcard load

In [6]:
bronze_rentals_df = to_bronze(
    read_raw_csv(
        f"{RAW}/rentals/rentals_*.csv",
        rentals_schema,
    )
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 8, Finished, Available, Finished, False)

#### 4. Write the four Bronze Delta tables

In [7]:
bronze_tables = {
    "bronze_customers": bronze_customers_df,
    "bronze_depots": bronze_depots_df,
    "bronze_rentals": bronze_rentals_df,
    "bronze_billing": bronze_billing_df,
}

for table_name, dataframe in bronze_tables.items():
    (
        dataframe.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Created: {table_name}")

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 9, Finished, Available, Finished, False)

Created: bronze_customers
Created: bronze_depots
Created: bronze_rentals
Created: bronze_billing


#### 5. Validate the row counts, schema, and lineage

In [8]:
expected_counts = {
    "bronze_customers": 602,
    "bronze_depots": 6,
    "bronze_rentals": 976,
    "bronze_billing": 787,
}

validation_results = []

for table_name, expected_count in expected_counts.items():
    df = spark.table(table_name)

    actual_count = df.count()

    lineage_columns_present = (
        "ingested_at" in df.columns
        and "source_file" in df.columns
    )

    raw_fields = [
        field
        for field in df.schema.fields
        if field.name not in {
            "ingested_at",
            "source_file",
        }
    ]

    non_string_raw_columns = [
        f"{field.name}: {field.dataType.simpleString()}"
        for field in raw_fields
        if field.dataType.simpleString() != "string"
    ]

    data_types = dict(df.dtypes)

    validation_results.append({
        "table_name": table_name,
        "expected_count": expected_count,
        "actual_count": actual_count,
        "count_matches": actual_count == expected_count,
        "raw_columns_are_strings":
            len(non_string_raw_columns) == 0,
        "lineage_columns_present":
            lineage_columns_present,
        "ingested_at_type":
            data_types.get("ingested_at"),
        "source_file_type":
            data_types.get("source_file"),
    })

    assert actual_count == expected_count, (
        f"{table_name}: expected {expected_count}, "
        f"found {actual_count}"
    )

    assert not non_string_raw_columns, (
        f"{table_name} has non-string source columns: "
        f"{non_string_raw_columns}"
    )

    assert lineage_columns_present, (
        f"{table_name} is missing lineage columns"
    )

    assert data_types["ingested_at"] == "timestamp"
    assert data_types["source_file"] == "string"


validation_df = spark.createDataFrame(
    validation_results
)

display(validation_df)

print("All Bronze validations passed.")

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9adc11a7-ac01-46e5-b457-69b2e443365e)

All Bronze validations passed.


#### 6. Prove that no cleaning occurred

In [9]:
display(
    spark.table("bronze_billing")
    .filter(
        F.col("AMOUNT_INR").contains("Rs.")
    )
    .select(
        "BILL_ID",
        "AMOUNT_INR",
        "ingested_at",
        "source_file",
    )
    .limit(10)
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d8738c0-c702-4647-b111-2311b2faa216)

In [10]:
display(
    spark.table("bronze_rentals")
    .groupBy("rental_type")
    .count()
    .orderBy("rental_type")
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7b4a3ed2-31c2-465b-bddd-3fd0b55f22b0)

#### 7. Prove the 10 June re-sent file

In [19]:
from pyspark.sql import functions as F

bronze_rentals = spark.table("bronze_rentals")

display(
    bronze_rentals
    .select("source_file")
    .distinct()
    .orderBy("source_file")
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dd478c55-1176-49a6-a191-c5459b35fc40)

In [20]:
june_10_rentals = (
    bronze_rentals
    .filter(
        F.lower(F.col("source_file"))
        .contains("rentals_2026-06-10")
    )
    .withColumn(
        "source_file_name",
        F.regexp_extract(
            F.col("source_file"),
            r"([^/\\]+\.csv)",
            1
        )
    )
)

display(
    june_10_rentals
    .groupBy("source_file_name")
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("rental_id").alias(
            "distinct_rental_ids"
        )
    )
    .orderBy("source_file_name")
)

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 992ce872-ca52-4176-a844-5722271a9d71)

#### 8. Prove that the rental IDs are exactly the same

In [22]:
original_ids = (
    june_10_rentals
    .filter(
        F.col("source_file_name")
        == "rentals_2026-06-10.csv"
    )
    .select("rental_id")
)

resend_ids = (
    june_10_rentals
    .filter(
        F.col("source_file_name")
        == "rentals_2026-06-10_resend.csv"
    )
    .select("rental_id")
)

only_in_original = original_ids.exceptAll(
    resend_ids
).count()

only_in_resend = resend_ids.exceptAll(
    original_ids
).count()

print("10 June resend validation passed.")
print(f"Only in original: {only_in_original}")
print(f"Only in resend:   {only_in_resend}")

assert only_in_original == 0
assert only_in_resend == 0

print("The original and resend contain the same rental IDs.")

StatementMeta(, 17bf72eb-ed9f-4f6a-9338-00901018a0f8, 24, Finished, Available, Finished, False)

10 June resend validation passed.
Only in original: 0
Only in resend:   0
The original and resend contain the same rental IDs.
